**Eye-Tracking & Neurocognitive Intelligence | Reading Comprehension & Neurocognitive Modelling | Presented by: Reza Saadatyar (2026)**

**04 | Sequential Modelling and Gaze–Text Fusion**

![Sequential Modelling and Gaze–Text Fusion — scientific workflow](../../images/04_sequential_and_multimodal_models.svg)

**Objective:** Compare masked CNN, GRU and Transformer fixation-sequence models with a text-only MLP and gaze–text fusion.

**Dataset:** Trial-level features and the local aligned fixation sequence cache from Notebook 02.

**Scope:** Predict the recorded correctness of a paragraph comprehension response from information available by the end of ordinary reading. No post-answer features, clinical inference or cognitive diagnosis.



**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess
import os
import sys

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
os.chdir(repo_path)
study_path = repo_path / "reading_comprehension_modelling"
if not (study_path / "code" / "src" / "onestop_io.py").is_file():
    raise FileNotFoundError("Update this checkout using git pull --ff-only origin main before running.")

# --------------------------- 3. Study Dependencies ---------------------------
# Existing compatible Colab packages are reused.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(study_path / "requirements.txt")], check=True)


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("reading_comprehension_modelling", sync_mode="off",
                   module_refresh="off")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

# ----------------- 4. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from onestop_io import validate_splits, validate_trial_index
from reading_features import load_sequences
from reading_models import neural_predictions, NEURAL
from reading_evaluation import metric

# ------------------------ 5. Output and Source Paths -------------------------
study = root / "reading_comprehension_modelling"
out_dir = study / "output"
out_dir.mkdir(parents=True, exist_ok=True)
raw_dir = root / "data" / "raw" / "onestop" / "ordinary"
processed = root / "data" / "processed" / "onestop"
processed.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
           label_size=11.5, label_weight="normal", tick_size=10, legend_size=10,
           figure_dpi=100, figure_size=(10, 5))


**02. Reconcile Ordered Fixation Sequences**

Load the trial feature table and local NPZ arrays. The cache reader matches ordered trial identities and checks sequence/mask dimensions. Padding is excluded from Training scaling and sequence pooling.

In [ ]:
# =================== 02. Reconcile Ordered Fixation Sequences ===================

identity = ["trial_id","participant_id","trial_index","article_batch",
            "article_key","paragraph_key","question_key","difficulty_level"]
table = pd.read_csv(out_dir/"02_trial_features.csv",dtype={c:str for c in identity})
index = pd.read_csv(out_dir/"01_trial_index.csv",dtype=str)
index["is_correct"] = index.is_correct.astype(int)
validate_trial_index(index,table)
splits = pd.read_csv(out_dir/"01_participant_splits.csv",dtype=str)
validate_splits(table,splits)
x,mask = load_sequences(processed/"02_fixation_sequences.npz",table)
print("Sequence candidates:",NEURAL)
print("Sequence shape:",x.shape)


**03. Train Sequence and Gaze–Text Models**

Use a compact masked 1D CNN, GRU and two-layer Transformer encoder, each with gaze-only and gaze–text fusion variants; a text-only MLP supplies a neural comparator. Text inputs are static paragraph statistics. AdamW, weighted binary cross-entropy and fixed 0.5 threshold are used. Retain the epoch with maximum Validation Macro F1, with up to 15 epochs and patience three. GPU is recommended.

In [ ]:
# =================== 03. Train Sequence and Gaze–Text Models ====================

neural, history, configs = neural_predictions(table,splits,x,mask,seed=17,epochs=15,patience=3)
print("Prediction rows:",len(neural))
display(history.tail())


**04. Inspect Validation Learning Curves**

Inspect the Validation trajectory for every candidate in each protocol. Early stopping means later epochs may be absent; do not fill missing epochs or score Test here.

In [ ]:
# ==================== 04. Inspect Validation Learning Curves ====================

fig, axes = plt.subplots(1,2,figsize=(13,5),sharey=True)
for ax,(protocol,g) in zip(axes,history.groupby("protocol")):
    for model,h in g.groupby("model"):
        ax.plot(h.epoch,h.validation_macro_f1,marker=".",label=model)
    ax.set(title=protocol,xlabel="Epoch",ylabel="Validation Macro F1")
    ax.legend(fontsize=7)
fig.tight_layout()
save_figure(fig,out_dir/"04_validation_curves.png")
plt.show()
v = neural[neural.role.eq("validation")].merge(
    table[["trial_id","is_correct"]],on="trial_id",validate="many_to_one")
validation = pd.DataFrame([dict(protocol=p,model=m,**metric(g.is_correct,g.p_correct))
                           for (p,m),g in v.groupby(["protocol","model"])])
display(validation.sort_values(["protocol","macro_f1"],ascending=[True,False]))


**05. Save Neural Prediction Scores**

Save the complete Validation and Test score table, selected-checkpoint Validation summaries and recorded training trajectories and fold-specific selected epochs, parameter counts and training settings. Model fitting is not repeated in the final notebook.

In [ ]:
# ====================== 05. Save Neural Prediction Scores =======================

neural.to_csv(out_dir/"04_neural_predictions.csv",index=False)
history.to_csv(out_dir/"04_training_history.csv",index=False)
configs.to_csv(out_dir/"04_model_configurations.csv",index=False)
validation.to_csv(out_dir/"04_validation_metrics.csv",index=False)
download_outputs_from_colab(out_dir,"04")
